In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from src.spark_session import get_spark_session

spark = get_spark_session("Delta-Inspection", driver_memory="2g")

# Tüm tablo path'leri
tables = {
    "🥉 Bronze - All transactions": "../delta_lake/bronze/transactions",
    "🥈 Silver - Purchases":          "../delta_lake/silver/transactions",
    "🥈 Silver - Cancellations":      "../delta_lake/silver/cancellations",
}

for name, path in tables.items():
    abs_path = os.path.abspath(path)
    if not os.path.exists(abs_path):
        print(f"❌ {name}: yok ({abs_path})")
        continue
    df = spark.read.format("delta").load(abs_path)
    count = df.count()
    print(f"{name}: {count:,} satır")

26/05/10 14:01:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 14:01:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


🥉 Bronze - All transactions: 10,100 satır
🥈 Silver - Purchases: 6,932 satır
🥈 Silver - Cancellations: 170 satır


In [2]:
# Detaylı şema bilgisi
print("\n🥈 Silver Purchases Şeması:")
df_silver = spark.read.format("delta").load(os.path.abspath("../delta_lake/silver/transactions"))
df_silver.printSchema()


🥈 Silver Purchases Şeması:
root
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- kullanici_ID: long (nullable = true)
 |-- olay_tipi: string (nullable = true)
 |-- ilgili_ID: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- country: string (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- total_price: double (nullable = true)
 |-- is_cancellation: boolean (nullable = true)



In [3]:
spark.stop()